volume transport code for anha12 -- in progress

In [1]:
"""
all_transport_calcuations.py
author: Tahya Weiss-Gibbons (adapted by Grace Kirkpatrick)

Calculate the volume, freshwater, heat and salt transport across a section for ANHA4
Need to provide the start and end model grid points, and a straight line in the model grid will be calculated from there
Outputs netCDF files with the output

"""
import math
import datetime
import numpy as np
import xarray as xr
import netCDF4 as nc

#dont try to open the corrupted files
def remove_bad_files(bad_path, runid, file_type, mdl_files):
    with open (bad_path+runid+"_bad_"+file_type+"_files.txt", "r") as file:
        bad_files = eval(file.readline())

    return [x for x in mdl_files if x not in bad_files]

#gets the list of grid points for a straight section between two points
#uses the bresenham line algorithm

def plotLineLow(x0, x1, y0, y1):

    print("low line!")

    ii = []
    jj = []

    dx = x1 -x0
    dy = y1 - y0
    yi = 1

    if dy < 0:
        yi = -1
        dy = -dy

    D = (2*dy) - dx
    y = y0

    for x in range(x0, x1):
        ii.append(x)
        jj.append(y)
        if D > 0:
            y = y+yi
            D = D+(2*(dy-dx))
            ii.append(x)
            jj.append(y)
        else:
            D = D+2*dy

    return ii, jj


def plotLineHigh(x0, x1, y0, y1):

    print("high line!")

    ii = []
    jj = []

    dx = x1 - x0
    dy = y1 - y0
    xi = 1

    if dx < 0:
        xi = -1
        dx = -dx
    D = (2*dx) - dy
    x = x0

    for y in range(y0, y1):
        ii.append(x)
        jj.append(y)
        if D > 0:
            x = x+xi
            D = D + (2*(dx-dy))
            ii.append(x)
            jj.append(y)
        else:
            D = D + 2*dx

    return ii, jj


def section_calculation(x0, x1, y0, y1):

    if abs(y1-y0) < abs(x1-x0):
        if x0 > x1:
            ii, jj = plotLineLow(x1, x0, y1, y0)
        else:
            ii, jj = plotLineLow(x0, x1, y0, y1)

    else:
            if y0 > y1:
                ii, jj = plotLineHigh(x1, x0, y1, y0)
            else:
                ii, jj = plotLineHigh(x0, x1, y0, y1)

    return ii, jj


def transport_calculations(runid, endyear, endmonth, endday, secname, n1, n2, n3, n4, startyear=2017, startmonth=1, startday=5):
    figs_path = '/mnt/storage6/grace/nwcorner/scripts/Figures/'
    path = "/mnt/storage6/myers/NEMO/ANHA12-" + runid + "/"
    
    other_path = '/mnt/storage6/grace/data/'
    bad_path = '/mnt/storage6/grace/nwcorner/bad_files/'

    start_time = datetime.date(startyear, startmonth, startday)

    end_time = datetime.date(endyear, endmonth, endday)


    #figure out all the dates we have model files
    delta = end_time - start_time
    times = []

    i = 0
    while i < delta.days+1:
        t = start_time + datetime.timedelta(days=i)
        if t.month == 2 and t.day == 29:
            t = datetime.date(t.year, 3, 1)
            i = i+6
        if t.year == 2019 and t.month == 6 and t.day == 19:
            i = i+4
        if t.year == 2019 and t.month == 12 and t.day == 30:
            i = i+6
        else:
            i = i+5
        times.append(t)
	
    #need both the u and v components of velocity
    mdl_files_v = []
    mdl_files_u = []
    mdl_files_t = []
    for t in times:
        mdl_files_v.append(path+"ANHA12-"+runid+"_y"+str(t.year)+"m"+str(t.month).zfill(2)+"d"+str(t.day).zfill(2)+"_gridV.nc")
        mdl_files_u.append(path+"ANHA12-"+runid+"_y"+str(t.year)+"m"+str(t.month).zfill(2)+"d"+str(t.day).zfill(2)+"_gridU.nc")
        mdl_files_t.append(path+"ANHA12-"+runid+"_y"+str(t.year)+"m"+str(t.month).zfill(2)+"d"+str(t.day).zfill(2)+"_gridT.nc")

    print("DONE TIMES")
    #mdl_files_v = remove_bad_files(bad_path, runid, 'v', mdl_files_v)
    #mdl_files_u = remove_bad_files(bad_path, runid, 'u', mdl_files_u)
    #mdl_files_t = remove_bad_files(bad_path, runid, 't', mdl_files_t)

    dv = xr.open_mfdataset(mdl_files_v, concat_dim='time_counter', data_vars='minimal',combine='nested', coords='minimal', compat='override')
    du = xr.open_mfdataset(mdl_files_u, concat_dim='time_counter', data_vars='minimal',combine='nested', coords='minimal', compat='override')
    dt = xr.open_mfdataset(mdl_files_t, concat_dim='time_counter', data_vars='minimal',combine='nested', coords='minimal', compat='override')

    #take the monthly average to deal with missing files
    #dv = dv.resample('ME', on='time_counter').mean()
    #du = du.resample('ME', on='time_counter').mean()
    #dt = dt.resample('ME', on='time_counter').mean()

    print(dv)
    print(du)
    print(dt)

    #read in the mask file
    mask_file = other_path+'ANHA12_mask.nc'
    mesh_file = other_path+'ANHA12_mesh_hgr.nc'
    mesh_file_z = other_path+'ANHA12_mesh_zgr.nc'
    
    mf = nc.Dataset(mask_file)
    mef = nc.Dataset(mesh_file)
    mez = nc.Dataset(mesh_file_z)
	
    nav_lon = np.array(mef.variables['nav_lon'])
    nav_lat = np.array(mef.variables['nav_lat'])
    umask = np.array(mf.variables['umask'])
    vmask = np.array(mf.variables['vmask'])
    tmask = np.array(mf.variables['tmask'])
    e1v = np.array(mef.variables['e1v'])[0,:,:]
    e2u = np.array(mef.variables['e2u'])[0,:,:]
    e3u = np.array(mez.variables['e3u'])[0,:, :,:]
    e3v = np.array(mez.variables['e3v'])[0,:, :,:]
    
    mf.close()
    mef.close()
    mez.close()

    #and finally the calcualted cell thickness
    #cell_thickness_u = np.load(other_path+'computed_u_thickness.npy')
    #cell_thickness_v = np.load(other_path+'computed_v_thickness.npy')

    #mask the data
    dv.coords['vmask'] = (('depthv', 'y', 'x'), vmask[0,:,:,:])
    du.coords['umask'] = (('depthu', 'y', 'x'), umask[0,:,:,:])
    dt.coords['tmask'] = (('deptht', 'y_grid_T', 'x_grid_T'), tmask[0,:,:,:])

    dv = dv.where(dv.vmask == 1)
    du = du.where(du.umask == 1)
    dt = dt.where(dt.tmask == 1)

    #need the list of i/j points for the section
    #can then use this to calculate the area of the section
    #and to see which velocity components are needed at each grid space
	
    #straight line

    section = secname # from func
    ii, jj = section_calculation(n1, n2, n3, n4) #from func

    t = du.dims['time_counter']
    total_volume = []
    #total_freshwater = []
    #total_heat = []
    #total_salt = []

    fw_trans = []

    lon = []
    lat = []
    comp = []
    d = []

    for n in range(0,len(ii)-1):
        i1 = int(ii[n])
        i2 = int(ii[n+1])
        j1 = int(jj[n])
        j2 = int(jj[n+1])
        lon.append(nav_lon[j1, i1])
        lat.append(nav_lat[j1,i1])

        #are we going south or west?
        negative=False
        if nav_lat[j1,i1] > nav_lat[j2,i2]: negative = True #south
        if nav_lon[j1,i1] > nav_lon[j2,i2]: negative = True #west

        #if negative == False: continue #just want the southward component for davis strait

        #so we should only either have i change or j change
        if j1 == j2:

            comp.append('v')
            
            vt = dv['vomecrty']
            v = vt.isel(y=j1, x=i1)
            v = v.rename('vel')
            v = v.rename({'depthv': 'depth'})

            cell_thickness = e3u[:,j1,i1]
            cell_width = np.ones(v.shape)*e1v[j1,i1]

            """

            #salinity and temperature are on the T grid, need them on v
            sal1 = dt['vosaline'].isel(y_grid_T=j1, x_grid_T=i1)
            sal2 = dt['vosaline'].isel(y_grid_T=j1, x_grid_T=i1+1)
            sal = (sal1+sal2)/2

            temp1 = dt['votemper'].isel(y_grid_T=j1, x_grid_T=i1)
            temp2 = dt['votemper'].isel(y_grid_T=j1, x_grid_T=i1+1)
            temp = (temp1+temp2)/2
            """

        elif i1 == i1:

            #change in x
            comp.append('u')

            vt = du['vozocrtx']
            v = vt.isel(y=j1, x=i1)
            v = v.rename('vel')
            v = v.rename({'depthu': 'depth'})
            
            cell_thickness = e3v[:,j1,i1]
            cell_width = np.ones(v.shape)*e2u[j1,i1]

            """

            #salinity and temperature are on the T grid, need them on u
            sal1 = dt['vosaline'].isel(y_grid_T=j1, x_grid_T=i1)
            sal2 = dt['vosaline'].isel(y_grid_T=j1+1, x_grid_T=i1)
            sal = (sal1+sal2)/2

            temp1 = dt['votemper'].isel(y_grid_T=j1, x_grid_T=i1)
            temp2 = dt['votemper'].isel(y_grid_T=j1+1, x_grid_T=i1)
            temp = (temp1+temp2)/2
            """

        #sal = sal.rename({'deptht': 'depth'})
        #temp = temp.rename({'deptht': 'depth'})

        #using a reference salinity of 34.8
        #fwc = (34.8-sal)/34.8

        if negative:
            print('negative!')
            vol = -v*cell_thickness*cell_width
            #fw = -v*fwc*cell_thickness*cell_width
            #heat = -v*temp*cell_thickness*cell_width
            #salt = -v*sal*cell_thickness*cell_width
        else:
            vol = v*cell_thickness*cell_width
            #fw = v*fwc*cell_thickness*cell_width
            #heat = v*temp*cell_thickness*cell_width
            #salt = v*sal*cell_thickness*cell_width

        vl = vol.sum(dim='depth')
        #f = fw.sum(dim='depth', skipna=True)
        #h = heat.sum(dim='depth')
        #s = salt.sum(dim='depth')

        fw_trans.append(vol)
        total_volume.append(vl)
        #total_freshwater.append(f)
        #total_heat.append(h)
        #total_salt.append(s)

    #now sum the data at each location    
    volume_transport = total_volume[0]
    #freshwater_transport = total_freshwater[0]
    #heat_transport = total_heat[0]
    #salt_transport = total_salt[0]

    for t in range(1, len(total_volume)):
        volume_transport = volume_transport+total_volume[t]
        #freshwater_transport = freshwater_transport+total_freshwater[t]
        #heat_transport = heat_transport+total_heat[t]
        #salt_transport = salt_transport+total_salt[t]

    #convert to sverdrups
    volume_transport = volume_transport*0.000001
    #freshwater_transport = freshwater_transport*0.000001
    #heat_transport = heat_transport*0.000001
    #salt_transport = salt_transport*0.000001

    #print(volume_transport.mean(dim='time_counter').values)

    #and output calcualted data to a netcdf file
    volume_transport.to_netcdf(figs_path+section+'_volume_transport_'+runid+'.nc')
    #freshwater_transport.to_netcdf(figs_path+section+'_freshwater_transport_'+runid+'.nc')
    #heat_transport.to_netcdf(figs_path+section+'_heat_transport_'+runid+'.nc')
    #salt_transport.to_netcdf(figs_path+section+'_salt_transport_'+runid+'.nc')

    dv.close()
    du.close()

In [52]:
if __name__ == "__main__":
    #transport_calculations(runid='EPM151', endyear=2022, endmonth=12, endday=31, secname = 'flemish', n1 = 57+125, n2=95+125, n3=70+225, n4=70+225 )
    #transport_calculations(runid='EPM151', endyear=2022, endmonth=12, endday=31, secname = 'lab_cur', n1 = 50+125, n2=70+125, n3=110+225, n4=121+225 )
    transport_calculations(runid='EXH006', endyear=2021, endmonth=12, endday=21, secname = 'pre_gb', n1 = 55+125, n2=85+125, n3=85+225, n4=85+225 )



DONE TIMES
<xarray.Dataset> Size: 288GB
Dimensions:       (y: 2400, x: 1632, time_counter: 361, depthv: 50)
Coordinates:
  * time_counter  (time_counter) float32 1kB 4.32e+05 8.64e+05 ... 1.568e+08
  * depthv        (depthv) float32 200B 0.494 1.541 ... 5.275e+03 5.728e+03
Dimensions without coordinates: y, x
Data variables:
    nav_lon       (y, x) float32 16MB dask.array<chunksize=(1200, 816), meta=np.ndarray>
    nav_lat       (y, x) float32 16MB dask.array<chunksize=(1200, 816), meta=np.ndarray>
    vomecrty      (time_counter, depthv, y, x) float32 283GB dask.array<chunksize=(1, 8, 400, 272), meta=np.ndarray>
    sometauy      (time_counter, y, x) float32 6GB dask.array<chunksize=(1, 1200, 816), meta=np.ndarray>
Attributes:
    Conventions:  GDT 1.3
    file_name:    ANHA12-EXH006_y2017m01d05_gridV.nc
    production:   OPA9.0 beta
    TimeStamp:    2018-JUL-23 13:39:26 GMT-0400
<xarray.Dataset> Size: 288GB
Dimensions:       (y: 2400, x: 1632, time_counter: 361, depthu: 50)
Coordin

/tmp/ipykernel_964315/517103686.py:172: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  nav_lon = np.array(mef.variables['nav_lon'])
/tmp/ipykernel_964315/517103686.py:173: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  nav_lat = np.array(mef.variables['nav_lat'])
/tmp/ipykernel_964315/517103686.py:174: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To lear

low line!


/tmp/ipykernel_964315/517103686.py:208: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  t = du.dims['time_counter']


In [24]:
    start_time = datetime.date(2022, 12, 1)

    end_time = datetime.date(2025, 12, 1)

    #figure out all the dates we have model files
    delta = end_time - start_time
    times = []

    i = 0
    while i < delta.days+1:
        t = start_time + datetime.timedelta(days=i)
        if t.month == 2 and t.day == 29:
            t = datetime.date(t.year, 3, 1)
            i = i+6
        if t.year == 2019 and t.month == 6 and day == 19:
            i = i+4
        else:
            i = i+5
        times.append(t)